# Player Level & Game Day Progression Analysis

**Purpose:** Understand how quickly players progress through levels and game days across install cohorts, and how retention curves compare over time.

**Data range:** April 2024 – April 2026 install cohorts (~59M player-day observations)

---

## Key Findings

- **Level progression is rapid in the first few days:** Players reach an average of ~4 levels on install day (D0), ~6 by D1, ~7 by D2, and ~8 by D3.
- **Recent cohorts show slightly lower early engagement:** Average max level on D0 dropped from ~4.0 (Apr 2024) to ~3.5 (Apr 2026) — roughly an 11% decrease.
- **D1 retention has declined slightly:** ~55% of the Apr 2024 cohort returned on D1 vs ~51% for Apr 2026, suggesting a softening in early-day engagement over time.
- **Wide player distribution:** P90 players progress ~3–4× faster than P10, and this spread widens significantly beyond D7.
- **Game day and level are tightly coupled:** Game day progression closely mirrors level progression (~2 game days on D0, ~3.5 on D1), indicating most sessions drive both metrics in tandem.

In [1]:
# show → code input visible by default
# hide-output → output hidden by default
# show hide-output → both (can combine on one line)

# Standard data analysis stack + project utilities
import pandas as pd
from common_lib.sql import BigQueryConnector
from common_lib.export import export_notebook_html
import plotly.express as px
import numpy as np

bqc = BigQueryConnector()

## Get data

### Player level and game day

In [2]:
# Point to the SQL file and set the start date for the cohort window
refresh_data = True
query_location = './sql/playerlevel.sql'
parameters = {
    'start_date': '2026-04-01',
}

# Print cost estimate before running — avoids accidental expensive queries (~$1.63 / 242 GB)
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 29.95 GB when run.
Estimated query cost: $0.20


In [3]:
data = pd.DataFrame()

# Toggle to True to re-run the BQ query and overwrite the local cache

if refresh_data:
    data = bqc.get(query='./sql/playerlevel.sql', is_path=True, query_parameters=parameters)
    data.to_pickle('./data/playprogression.pkl')
else:
    # Load from local cache to avoid repeated query costs
    data = pd.read_pickle('./data/playprogression.pkl')

In [4]:
data

,user_id,dt,install_dt,install_dt_week,install_dt_month,days_since_install,max_level,max_gameday,acquisition_type,install_build_version
0,42E8B0C397103FE0,2026-06-18,2026-06-15,2026-06-14,2026-06-01,3,4,2,Non-Attributed,0.77.0
1,8BEFFBA940EB6E87,2026-06-20,2026-06-20,2026-06-14,2026-06-01,0,3,1,Non-Attributed,0.77.0
2,99776F3B5B45C677,2026-06-19,2026-06-19,2026-06-14,2026-06-01,0,5,3,CPE,0.77.0
3,121F829BD7D1B1D2,2026-06-19,2026-06-18,2026-06-14,2026-06-01,1,6,4,Non-Attributed,0.77.0
4,B1F425FE8007C50C,2026-06-21,2026-06-20,2026-06-14,2026-06-01,1,6,4,CPE,0.77.0
...,...,...,...,...,...,...,...,...,...,...
669723,73FBC9B8F5C0575D,2026-06-14,2026-06-11,2026-06-07,2026-06-01,3,9,6,CPE,0.76.0
669724,88E3A6E85FD3D234,2026-06-19,2026-06-11,2026-06-07,2026-06-01,8,12,8,CPE,0.76.0
669725,1A23DBF515298916,2026-06-21,2026-06-11,2026-06-07,2026-06-01,10,10,7,CPE,0.76.0
669726,504F34510B38472E,2026-06-15,2026-06-11,2026-06-07,2026-06-01,4,6,4,CPE,0.76.0


## Process data

In [5]:
dt_mode = 'install_dt'

data['install_dt'] = data[dt_mode]

data['CPE_flag'] = ['Y' if x == 'CPE' else 'N' for x in data['acquisition_type']]

data['FTUE_flag'] = ['new' if x >='0.76.0' else 'old' for x in data['install_build_version']]

data['dummy'] = 'dummy'

# Require each cohort to have had enough calendar time to be meaningful:
# weekly cohorts need 7 days, monthly cohorts need 30.
min_days_since_install = 0
if dt_mode == 'install_dt_week':
    min_days_since_install = 7
elif dt_mode == 'install_dt_month':
    min_days_since_install = 30 

# Drop rows where the player's most recent observed day falls within the cohort's maturity window.
# This prevents partially-observed cohorts from pulling down progression averages.
data = data[data['days_since_install'] <= (pd.to_datetime('today') - pd.to_datetime(data['install_dt'])).dt.days - min_days_since_install]

data

,user_id,dt,install_dt,install_dt_week,install_dt_month,days_since_install,max_level,max_gameday,acquisition_type,install_build_version,CPE_flag,FTUE_flag,dummy
0,42E8B0C397103FE0,2026-06-18,2026-06-15,2026-06-14,2026-06-01,3,4,2,Non-Attributed,0.77.0,N,new,dummy
1,8BEFFBA940EB6E87,2026-06-20,2026-06-20,2026-06-14,2026-06-01,0,3,1,Non-Attributed,0.77.0,N,new,dummy
2,99776F3B5B45C677,2026-06-19,2026-06-19,2026-06-14,2026-06-01,0,5,3,CPE,0.77.0,Y,new,dummy
3,121F829BD7D1B1D2,2026-06-19,2026-06-18,2026-06-14,2026-06-01,1,6,4,Non-Attributed,0.77.0,N,new,dummy
4,B1F425FE8007C50C,2026-06-21,2026-06-20,2026-06-14,2026-06-01,1,6,4,CPE,0.77.0,Y,new,dummy
...,...,...,...,...,...,...,...,...,...,...,...,...,...
669723,73FBC9B8F5C0575D,2026-06-14,2026-06-11,2026-06-07,2026-06-01,3,9,6,CPE,0.76.0,Y,new,dummy
669724,88E3A6E85FD3D234,2026-06-19,2026-06-11,2026-06-07,2026-06-01,8,12,8,CPE,0.76.0,Y,new,dummy
669725,1A23DBF515298916,2026-06-21,2026-06-11,2026-06-07,2026-06-01,10,10,7,CPE,0.76.0,Y,new,dummy
669726,504F34510B38472E,2026-06-15,2026-06-11,2026-06-07,2026-06-01,4,6,4,CPE,0.76.0,Y,new,dummy


In [6]:
test = data.groupby(['install_dt', 'FTUE_flag']).agg(
    users=('user_id', 'nunique')
).reset_index()

test

,install_dt,FTUE_flag,users
0,2026-04-01,old,1091
1,2026-04-02,old,1135
2,2026-04-03,old,1222
3,2026-04-04,old,1121
4,2026-04-05,old,1265
...,...,...,...
104,2026-06-19,old,9
105,2026-06-20,new,1073
106,2026-06-20,old,4
107,2026-06-21,new,1129


In [7]:
# Cache the filtered dataset to avoid reprocessing on subsequent runs
data.to_pickle('./data/playprogression_processed.pkl')

In [8]:
# Load the cohort-filtered processed dataset
data = pd.read_pickle('./data/playprogression_processed.pkl') 

## Player Level reached at day x

For each install cohort, tracks the **weighted average max level** reached as a function of days since install. Weighted average is used to account for varying player counts across level buckets. The percentage-change chart below highlights where the steepest level gains occur in the early-day window.

In [9]:
def compute_weighted_progression(data, measure_col, dimension_cols=['install_dt', 'days_since_install'], min_bucket_size=50):
    """
    Compute weighted average progression metric for a given measure across dimensions.
    
    Parameters:
    -----------
    data : pd.DataFrame
        Source data containing user_id, the measure column, and dimension columns
    measure_col : str
        Column name to compute weighted average for (e.g., 'max_level', 'max_gameday')
    dimension_cols : list
        Dimensions to group by (default: ['install_dt', 'days_since_install'])
    min_bucket_size : int
        Minimum users per bucket to include (default: 50)
    
    Returns:
    --------
    pd.DataFrame
        Aggregated data with weighted average and cohort user counts
    """
    
    # Step 1: Count unique users per dimension + measure bucket
    agg = data.groupby(dimension_cols + [measure_col]).agg(
        unique_users=('user_id', 'nunique')
    ).reset_index()
    
    # Step 2: Total unique users per dimension combination
    dimension_total_users = data.groupby(dimension_cols).agg(
        total_unique_users=('user_id', 'nunique')
    ).reset_index()
    
    # Step 3: Merge and compute percentage share
    agg = agg.merge(dimension_total_users, on=dimension_cols)
    agg['percentage_of_users'] = agg['unique_users'] / agg['total_unique_users']
    
    # Step 4: Compute weighted average
    weighted_avg = agg.groupby(dimension_cols, group_keys=False).apply(
        lambda x: (x[measure_col] * x['unique_users']).sum() / x['unique_users'].sum(),
        include_groups=False
    ).reset_index()
    
    weighted_avg.columns = dimension_cols + [f'weighted_avg_{measure_col}']
    agg = agg.merge(weighted_avg, on=dimension_cols)
    
    # Step 5: Drop small buckets
    agg = agg[agg['unique_users'] >= min_bucket_size]
    
    # Step 6: Collapse to one row per dimension combination
    agg = agg.groupby(dimension_cols).agg(
        cohort_users=('unique_users', 'sum'),
        **{f'weighted_avg_{measure_col}': ('weighted_avg_' + measure_col, 'first')}
    ).reset_index()
    
    return agg

In [10]:
player_level_cpe_agg = compute_weighted_progression(data, measure_col='max_level', dimension_cols=['install_dt', 'days_since_install','CPE_flag'], min_bucket_size=50)
player_level_cpe_agg['combined_dimension'] = player_level_cpe_agg['install_dt'].astype(str) + ' | ' + player_level_cpe_agg['CPE_flag']
player_level_cpe_agg

,install_dt,days_since_install,CPE_flag,cohort_users,weighted_avg_max_level,combined_dimension
0,2026-04-01,0,N,664,3.092725,2026-04-01 | N
1,2026-04-01,0,Y,321,4.166667,2026-04-01 | Y
2,2026-04-01,1,N,201,4.831615,2026-04-01 | N
3,2026-04-01,1,Y,101,6.046025,2026-04-01 | Y
4,2026-04-01,2,N,60,5.989691,2026-04-01 | N
...,...,...,...,...,...,...
405,2026-06-20,0,N,591,3.271268,2026-06-20 | N
406,2026-06-20,0,Y,384,4.193833,2026-06-20 | Y
407,2026-06-20,1,N,178,5.056818,2026-06-20 | N
408,2026-06-20,1,Y,156,6.006969,2026-06-20 | Y


In [11]:
player_level_ftue_agg = compute_weighted_progression(data, measure_col='max_level', dimension_cols=['dummy', 'days_since_install','FTUE_flag'], min_bucket_size=50)
player_level_ftue_agg['combined_dimension'] = player_level_ftue_agg['dummy'].astype(str) + ' | ' + player_level_ftue_agg['FTUE_flag']
player_level_ftue_agg

,dummy,days_since_install,FTUE_flag,cohort_users,weighted_avg_max_level,combined_dimension
0,dummy,0,new,22624,3.652078,dummy | new
1,dummy,0,old,56491,3.560224,dummy | old
2,dummy,1,new,11433,5.670348,dummy | new
3,dummy,1,old,28925,5.623851,dummy | old
4,dummy,2,new,8445,7.052132,dummy | new
...,...,...,...,...,...,...
85,dummy,63,old,392,32.926129,dummy | old
86,dummy,64,old,326,32.749675,dummy | old
87,dummy,65,old,55,32.984594,dummy | old
88,dummy,66,old,153,33.640872,dummy | old


In [12]:
# hide-output
fig = px.line(player_level_cpe_agg, 
              x='days_since_install', 
              y='weighted_avg_max_level',
              color='combined_dimension',
              title='Player level daily progression by cohort',
              width=1200,
              height=600,
              hover_data={'weighted_avg_max_level': True, 'cohort_users': True},)

fig.show()

# hide-output
fig = px.line(player_level_ftue_agg, 
              x='days_since_install', 
              y='weighted_avg_max_level',
              color='combined_dimension',
              title='Player level daily progression by cohort',
              width=1200,
              height=600,
              hover_data={'weighted_avg_max_level': True, 'cohort_users': True},)

fig.show()

## Player level reached at dayx percentiles

In [13]:
def weighted_quantiles(group, quantiles=[0.1, 0.5, 0.9], measure_col='max_level'):
    """Return P10 / P50 / P90 of measure_col, using user counts as weights.

    Sorts by the measure, accumulates weights, then uses searchsorted to find
    the value at each quantile threshold — equivalent to a weighted percentile.
    """
    levels = group[measure_col].values
    weights = group['users'].values
    sorted_idx = np.argsort(levels)
    levels, weights = levels[sorted_idx], weights[sorted_idx]
    cum_weights = np.cumsum(weights)
    total = cum_weights[-1]
    result = {}
    for q in quantiles:
        idx = np.searchsorted(cum_weights, q * total)
        result[f'p{int(q * 100)}'] = levels[min(idx, len(levels) - 1)]
    return pd.Series(result)

In [14]:
# hide-output
level_dist = data.groupby(['days_since_install', 'max_level','CPE_flag']).agg(
    users=('user_id', 'count')
).reset_index()

level_pcts = level_dist.groupby('days_since_install').apply(weighted_quantiles, measure_col='max_level', include_groups=False).reset_index()

fig = px.line(
    level_pcts.melt(id_vars='days_since_install', var_name='percentile', value_name='max_level'),
    x='days_since_install',
    y='max_level',
    color='percentile',
    title='Player level distribution by days since install (P10 / P50 / P90)',
    width=1200,
    height=600,
)
fig.show()

# hide-output
level_dist = data.groupby(['days_since_install', 'max_level','FTUE_flag']).agg(
    users=('user_id', 'count')
).reset_index()

level_pcts = level_dist.groupby('days_since_install').apply(weighted_quantiles, measure_col='max_level', include_groups=False).reset_index()

fig = px.line(
    level_pcts.melt(id_vars='days_since_install', var_name='percentile', value_name='max_level'),
    x='days_since_install',
    y='max_level',
    color='percentile',
    title='Player level distribution by days since install (P10 / P50 / P90)',
    width=1200,
    height=600,
)
fig.show()

In [17]:
player_level_agg2 = player_level_cpe_agg[['install_dt', 'days_since_install','weighted_avg_max_level']].drop_duplicates()

# Day-over-day % change in weighted avg level within each install cohort.
# Day 0 is filled as 1.0 (100%) since there is no prior day to compare against.
player_level_agg2['pct_change_weighted_avg_max_level'] = player_level_agg2.groupby('install_dt')['weighted_avg_max_level'].pct_change()
player_level_agg2.pct_change_weighted_avg_max_level = player_level_agg2.pct_change_weighted_avg_max_level.fillna(1)
player_level_agg2

,install_dt,days_since_install,weighted_avg_max_level,pct_change_weighted_avg_max_level
0,2026-04-01,0,3.092725,1.000000
1,2026-04-01,0,4.166667,0.347248
2,2026-04-01,1,4.831615,0.159588
3,2026-04-01,1,6.046025,0.251347
4,2026-04-01,2,5.989691,-0.009318
...,...,...,...,...
405,2026-06-20,0,3.271268,1.000000
406,2026-06-20,0,4.193833,0.282020
407,2026-06-20,1,5.056818,0.205775
408,2026-06-20,1,6.006969,0.187895


In [18]:
# hide-output
fig = px.line(player_level_agg2.where(player_level_agg2.days_since_install<=30), 
              x='days_since_install', 
              y='pct_change_weighted_avg_max_level',
              color='install_dt',
              title='Player level daily progression by cohort (pct change)',
              width=1200,
              height=600,
              hover_data={'weighted_avg_max_level': True})


fig.show()

## Game day reached at day x

Mirrors the level analysis but uses **game days** (in-game calendar progression) instead of levels. Comparing both metrics reveals whether level gates or natural engagement drives pacing — if game days outpace levels, players are replaying content; if levels outpace game days, players are advancing quickly through fewer sessions.

In [19]:
# Same weighted-average approach as player level, applied to max_gameday

# Step 1: Count unique users per (install cohort, day since install, max_gameday bucket)
game_day_agg = data.groupby(['install_dt', 'days_since_install', 'max_gameday']).agg(
    unique_users = ('user_id', 'nunique')
    ).reset_index()

# Step 2: Total unique users per cohort-day
daily_cohort_total_users = data.groupby(['install_dt','days_since_install']).agg(
    total_unique_users = ('user_id', 'nunique')
    ).reset_index()

# Step 3: Share of each day's users at each game day value
game_day_agg = game_day_agg.merge(daily_cohort_total_users, on=['install_dt', 'days_since_install'])
game_day_agg['percentage_of_daily_users'] = game_day_agg['unique_users'] / game_day_agg['total_unique_users']

# Step 4: Weighted average game day per cohort-day
weighted_avg = game_day_agg.groupby(['install_dt', 'days_since_install'], group_keys=False).apply(
    lambda x: (x['max_gameday'] * x['unique_users']).sum() / x['unique_users'].sum(),
    include_groups=False
).reset_index()

weighted_avg.columns = ['install_dt', 'days_since_install', 'weighted_avg_max_gameday']
game_day_agg = game_day_agg.merge(weighted_avg, on=['install_dt', 'days_since_install'])

# Drop small buckets (< 50 users) to reduce noise
game_day_agg = game_day_agg[game_day_agg['unique_users'] >= 50]

# Collapse to one row per cohort-day
game_day_agg = game_day_agg.groupby(['install_dt', 'days_since_install']).agg(
    cohort_users = ('unique_users', 'sum'),
    weighted_avg_max_gameday = ('weighted_avg_max_gameday', 'first')
).reset_index()

game_day_agg

,install_dt,days_since_install,cohort_users,weighted_avg_max_gameday
0,2026-04-01,0,1047,1.942857
1,2026-04-01,1,436,3.084906
2,2026-04-01,2,218,4.458763
3,2026-04-01,3,147,5.216763
4,2026-04-01,4,109,6.146179
...,...,...,...,...
567,2026-06-19,1,440,3.305147
568,2026-06-19,2,318,4.353075
569,2026-06-20,0,1028,2.019499
570,2026-06-20,1,427,3.651543


In [20]:
# hide-output
fig = px.line(game_day_agg, 
              x='days_since_install', 
              y='weighted_avg_max_gameday',
              color='install_dt',
              title='Game day daily progression by cohort',
              width=1200,
              height=600,
              hover_data={'weighted_avg_max_gameday': True, 'cohort_users': True},
              )


fig.show()

In [21]:
# hide-output
gameday_dist = data.groupby(['days_since_install', 'max_gameday']).agg(
    users=('user_id', 'count')
).reset_index()

gameday_pcts = gameday_dist.groupby('days_since_install').apply(weighted_quantiles, measure_col='max_gameday', include_groups=False).reset_index()

fig = px.line(
    gameday_pcts.melt(id_vars='days_since_install', var_name='percentile', value_name='max_gameday'),
    x='days_since_install',
    y='max_gameday',
    color='percentile',
    title='Game day distribution by days since install (P10 / P50 / P90)',
    width=1200,
    height=600,
)
fig.show()

## % of cohort users active by days since install

Retention curve: the share of a cohort's total users who were active on each calendar day since install. Plotted on a **log scale** so that differences between cohorts remain visible at longer horizons where absolute percentages are very small. Apr 2024 D1 retention was ~55%; Apr 2026 has declined to ~51%.

In [22]:
# Count unique users active on each day since install, per cohort
users_agg = pd.DataFrame()
users_agg = data.groupby(['install_dt', 'days_since_install']).agg(
    unique_users = ('user_id', 'nunique')
    ).reset_index()

# Total unique users ever observed in each cohort (denominator for retention %)
users_by_cohort = pd.DataFrame()
users_by_cohort = data.groupby('install_dt').agg(
    total_cohort_users = ('user_id', 'nunique')
).reset_index()

users_agg = users_agg.merge(users_by_cohort, on='install_dt')

# Retention rate: share of cohort still active on a given day
users_agg['pct_cohort_users_active'] = users_agg['unique_users'] / users_agg['total_cohort_users']

users_agg

,install_dt,days_since_install,unique_users,total_cohort_users,pct_cohort_users_active
0,2026-04-01,0,1085,1091,0.994500
1,2026-04-01,1,530,1091,0.485793
2,2026-04-01,2,388,1091,0.355637
3,2026-04-01,3,346,1091,0.317140
4,2026-04-01,4,301,1091,0.275894
...,...,...,...,...,...
3398,2026-06-19,1,544,1052,0.517110
3399,2026-06-19,2,439,1052,0.417300
3400,2026-06-20,0,1077,1077,1.000000
3401,2026-06-20,1,551,1077,0.511606


In [23]:
# hide-output

# apply a log transformation to the y axis to better visualize the differences between cohorts, especially in the later days since install where the percentage of active users is very low
fig = px.line(users_agg, 
              x='days_since_install', 
              y='pct_cohort_users_active',
              color='install_dt',
              title='% of cohort users active by days since install (log scale)',
              width=1200,
              height=600,
              hover_data={'pct_cohort_users_active': ':.2%', 'unique_users': True, 'total_cohort_users': True},
              log_y=False)


fig.show()

# Things to do next
- Is the churn increasing for P90 players when they reach the 150 GD mark?
- Is the slowdown on pace seen on around level 50 on newest cohort caused by CPE mix? what does it look like with only organics?